# Biochemistry Grapn Neural Network (GNN) Example
We will apply a GCN and a edge-conditioned MPNN to a dataset of protein structures annotated with Enzyme Commission (EC) numbers. EC numbers specify what reaction an enzyme classifies. This is a multiclass classification problem. The inspiration for this classification task is this article: https://www.nature.com/articles/s41467-021-23303-9 which trained a model to predict EC numbers and gene ontology (GO) terms. The dataset is a smaller version of the dataset found here: https://zenodo.org/records/7593591. Note you will need the whole folder labeled ec_prediction uploaded to Google colab, this contains both the labels in metadata.csv as well as all of the protein structures as pdb files.

# Install Packages
Only run if you do not have packages installed

In [ ]:
%pip install biopython
%pip install pandas
%pip install scikit-learn
%pip install matplotlib
%pip install torch torchvision torchaudio torch-geometric

# Set seed

In [5]:
import copy
import random
import os
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from Bio.PDB import PDBParser
from Bio.SeqUtils import seq1
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torch import nn
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, NNConv, MessagePassing, global_mean_pool
from torch_geometric.utils import softmax, to_dense_batch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cpu


# Load structures and construct graphs for protein structures

In [19]:
#input data location and model hyperparameters
metadata = "ec_prediction/metadata.csv"
pdb_dir = "ec_prediction/"
neighbors = 12
hidden = 32
batch_size = 32
epochs = 20

#information for constructing graph
AMINO_ACIDS = "ACDEFGHIKLMNPQRSTVWY"
AA_INDEX = {aa: i for i, aa in enumerate(AMINO_ACIDS)}
# Fixed, 2D sequence-based features and 3D C-alpha graph geometry
NODE_DIM = len(AMINO_ACIDS) + 1  # amino acid one-hot + normalized sequence position
RBF_CENTERS = np.linspace(2.0, 20.0, 16, dtype=np.float32)
EDGE_DIM = len(RBF_CENTERS) + 2  # distance RBF, signed sequence offset, adjacent flag

def locate_pdb(row, pdb_dir):
    """Prefer direct PDBID-CHAIN.pdb; allow metadata path's basename."""
    identifier = str(row["protein_id"]).strip()
    filename = f"{identifier}.pdb"
    meta_name = str(row.get("structure_file", "")).replace("\\", "/").split("/")[-1]
    candidates = [pdb_dir + "/" + filename]
    if meta_name and meta_name != filename:
        candidates.append(pdb_dir / meta_name)
    for path in candidates:
        if os.path.isfile(path):
            return path
    return None

#this method will make the graph for a PDB file
def make_graph(row, pdb_dir, parser, k=12):
    path = locate_pdb(row, pdb_dir)
    if path is None:
        return None, "missing_file"
    try:
        structure = parser.get_structure(str(row["protein_id"]), str(path))
        model = next(structure.get_models())
        chain_id = str(row["chain"]).strip()
        if chain_id not in model:
            return None, "missing_chain"
        chain = model[chain_id]
        residues = []
        coords = []
        for res in chain:
            #only look at alpha carbons to represent amino acid
            if res.id[0] != " " or "CA" not in res:
                continue
            aa = seq1(res.resname, custom_map={"MSE": "M"})
            if aa not in AA_INDEX:
                continue
            coord = np.asarray(res["CA"].coord, dtype=np.float32)
            if not np.isfinite(coord).all():
                continue
            residues.append((res, aa))
            coords.append(coord)
        n = len(residues)
        if n < 2:
            return None, "too_few_residues"

        x = np.zeros((n, NODE_DIM), dtype=np.float32)
        for i, (_, aa) in enumerate(residues):
            x[i, AA_INDEX[aa]] = 1.0
            x[i, -1] = i / max(n - 1, 1)

        coords = np.stack(coords)
        # Pairwise Euclidean distances are invariant to translation/rotation.
        distances = np.linalg.norm(coords[:, None, :] - coords[None, :, :], axis=-1)
        np.fill_diagonal(distances, np.inf)
        
        # amino acids are connected by edges to the k closest amino acid
        neighbors = min(k, n - 1)
        # k nearest spatial neighbors for each residue; symmetrize by including
        # both edge directions. Consecutive observed residues are also connected.
        edges = set()
        for i in range(n):
            nearest = np.argpartition(distances[i], neighbors - 1)[:neighbors]
            for j in nearest:
                j = int(j)
                if i != j:
                    edges.add((i, j))
                    edges.add((j, i))
        for i in range(n - 1):
            # Do not assume index-adjacent CA residues are necessarily close in 3D.
            edges.add((i, i + 1))
            edges.add((i + 1, i))

        pairs = sorted(edges)
        edge_index = np.asarray(pairs, dtype=np.int64).T
        attrs = []
        for i, j in pairs:
            d = float(distances[i, j])
            rbf = np.exp(-0.5 * ((d - RBF_CENTERS) / 1.5) ** 2)
            delta = j - i
            attrs.append(np.concatenate([
                rbf,
                np.array([np.clip(delta, -32, 32) / 32.0, float(abs(delta) == 1)], dtype=np.float32),
            ]))
        graph = Data(
            x=torch.tensor(x, dtype=torch.float32),
            edge_index=torch.tensor(edge_index, dtype=torch.long),
            edge_attr=torch.tensor(np.asarray(attrs), dtype=torch.float32),
            y=torch.tensor([int(row["class_id"])], dtype=torch.long),
        )
        return graph, "ok"
    except Exception as exc:
        print(f"Could not parse {path}: {exc}")
        return None, "parse_error"

#read and process data
df = pd.read_csv(metadata).dropna(subset=["protein_id", "chain", "label"])
classes = sorted(df["label"].unique().tolist())
class_map = {label: i for i, label in enumerate(classes)}
df["class_id"] = df["label"].map(class_map)
print("EC subclasses:", class_map)
print("Metadata rows:", len(df))

parser = PDBParser(QUIET=True)
graphs, kept_rows, skipped = [], [], Counter()
for _, row in df.iterrows():
    graph, status = make_graph(row, pdb_dir, parser, k=neighbors)
    if graph is None:
        skipped[status] += 1
    else:
        graphs.append(graph)
        kept_rows.append(row.to_dict())
print("Loaded structures:", len(graphs), "skipped:", dict(skipped))

labels = [int(g.y.item()) for g in graphs]
print("Usable per class:", {c: labels.count(i) for i, c in enumerate(classes)})
indices = np.arange(len(graphs))
# Split after excluding missing/invalid structures, stratified by EC class.
train_idx, val_idx = train_test_split(indices, test_size=0.20,
                                     random_state=42, stratify=labels)

def loader(idx, shuffle=False):
    return DataLoader([graphs[i] for i in idx], batch_size=batch_size, shuffle=shuffle)
train_loader, val_loader = loader(train_idx, True), loader(val_idx)
print("Train/val sizes:", len(train_idx), len(val_idx))
    

EC subclasses: {'1.1.1.-': 0, '2.1.1.-': 1, '2.3.1.-': 2, '3.2.1.-': 3, '3.4.21.-': 4}
Metadata rows: 998
Loaded structures: 998 skipped: {}
Usable per class: {'1.1.1.-': 200, '2.1.1.-': 199, '2.3.1.-': 200, '3.2.1.-': 200, '3.4.21.-': 199}
Train/val sizes: 798 200


# Edge-conditioned MPNN with attention over neighboring atoms
Each bond generates an edge-specific transformation of the source atom features. An attention score is then learned for each directed bond and normalized with a softmax over incoming neighbors of the destination atom. This replaces the equal-weight sum in NNConv. Both the message and the attention score use bond features.

In [21]:
#we need a new class to incorporate attention into the message passing
class EdgeAttentionConv(MessagePassing):
    """Bond-conditioned messages with softmax attention across incoming edges."""
    def __init__(self, in_channels, out_channels, edge_dim, hidden_dim=32):
        super().__init__(aggr="add", node_dim=0) #will sum messages
        self.in_channels = in_channels
        self.out_channels = out_channels

        # Each bond produces a matrix that transforms the source atom embedding.
        self.edge_network = nn.Sequential(
            nn.Linear(edge_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, in_channels * out_channels)
        )

        # Attention score depends on source, destination, AND bond features.
        self.attention_network = nn.Sequential(
            nn.Linear(2 * in_channels + edge_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
        #root network performs the update on the atom receiving messages
        #this will be added to the message
        self.root = nn.Linear(in_channels, out_channels)

    def forward(self, x, edge_index, edge_attr):
        # propagate() collects messages from source nodes and aggregates at targets
        neighbor_messages = self.propagate(
            edge_index, x=x, edge_attr=edge_attr, size=(x.size(0), x.size(0))
        )
        return neighbor_messages + self.root(x)

    def message(self, x_i, x_j, edge_attr, index, ptr, size_i):
        # x_j: source node, x_i: destination node, one row per directed bond
        weights = self.edge_network(edge_attr).view(
            -1, self.in_channels, self.out_channels
        )
        messages = torch.bmm(x_j.unsqueeze(1), weights).squeeze(1)

        logits = self.attention_network(
            torch.cat([x_i, x_j, edge_attr], dim=-1)
        ).squeeze(-1)
        # normalize over the *incoming edges of each destination node*
        alpha = softmax(logits, index, ptr=ptr, num_nodes=size_i)
        return messages * alpha.unsqueeze(-1)
    
class EdgeMPNNClassifier(nn.Module):
    def __init__(self, node_dim=NODE_DIM, edge_dim=EDGE_DIM, hidden_dim=32, n_classes=5):
        super().__init__()
        #use our custom EdgeAttentionConv as the layers in the GNN
        self.conv1 = EdgeAttentionConv(node_dim, hidden_dim, edge_dim, hidden_dim)
        self.conv2 = EdgeAttentionConv(hidden_dim, hidden_dim, edge_dim, hidden_dim)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_classes)
        )

    def forward(self, data):
        x = data.x
        edge_index = data.edge_index
        edge_attr = data.edge_attr

        x = torch.relu(
            self.conv1(x, edge_index, edge_attr)
        )

        x = torch.relu(
            self.conv2(x, edge_index, edge_attr)
        )

        x = global_mean_pool(x, data.batch)

        return self.classifier(x)

# Global graph Transformer (hybrid local-message passing/global transformer)
Unlike the preceding MPNN, whose attention is restricted to nearby neighbors, this model uses global multi-head self-attention across every amino acid in each protein. A bond-aware NNConv first provides local structural information; then a standard Transformer encoder lets distant amino acids interact directly. Padding masks ensure atoms in different proteins or padding positions cannot attend to one another. 

In [ ]:
class GlobalGraphTransformerModel(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim=32, num_heads=4,
                 num_transformer_layers=2, dropout=0.1,n_classes=5):
        super().__init__()
        if hidden_dim % num_heads != 0:
            raise ValueError("hidden_dim must be divisible by num_heads")
        #local message passing part
        # Local bond-conditioned aggregation encodes the molecular connectivity.
        edge_nn = nn.Sequential(
            nn.Linear(edge_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, node_dim * hidden_dim)
        )
        self.local_conv = NNConv(node_dim, hidden_dim, nn=edge_nn, aggr="add")
        
        #transformer part
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=hidden_dim * 4,
            dropout=dropout,
            batch_first=True,
            activation="relu"
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=num_transformer_layers,
            enable_nested_tensor=False
        )
        
        #regression head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, n_classes)
        )

    def forward(self, data):
        x = torch.relu(self.local_conv(data.x, data.edge_index, data.edge_attr))

        # [total_atoms, hidden_dim] -> [batch, max_atoms, hidden_dim]
        dense_x, mask = to_dense_batch(x, data.batch)

        # Every real atom attends to all real atoms in its own molecule.
        # The padding mask is True for positions that should be ignored.
        dense_x = self.transformer(
            dense_x, src_key_padding_mask=~mask
        )

        # Masked mean pool gives one vector per molecule.
        dense_x = dense_x.masked_fill(~mask.unsqueeze(-1), 0.0)
        graph_embeddings = dense_x.sum(dim=1) / mask.sum(
            dim=1, keepdim=True
        ).clamp(min=1)
        return self.classifier(graph_embeddings)

# Training models

In [ ]:
def run_epoch(model, loader, device, optimizer=None):
    training = optimizer is not None
    model.train(training)
    loss_sum, correct, total = 0.0, 0, 0
    all_true, all_pred = [], []
    for batch in loader:
        batch = batch.to(device)
        with torch.set_grad_enabled(training):
            logits = model(batch)
            labels = batch.y.view(-1)
            loss = nn.functional.cross_entropy(logits, labels)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        preds = logits.argmax(dim=-1)
        count = labels.numel()
        loss_sum += loss.item() * count
        correct += (preds == labels).sum().item()
        total += count
        all_true.extend(labels.detach().cpu().tolist())
        all_pred.extend(preds.detach().cpu().tolist())
    return {"loss": loss_sum / total, "accuracy": correct / total,
            "f1_macro": f1_score(all_true, all_pred, average="macro", zero_division=0),
            "y_true": all_true, "y_pred": all_pred}


def train_model(model, train_loader, val_loader, device, epochs=200, lr=1e-3):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = {key: [] for key in ("train_loss", "val_loss", "train_accuracy", "val_accuracy")}
    best_weights = copy.deepcopy(model.state_dict())
    best_loss, best_epoch, stale = float("inf"), 0, 0
    for epoch in range(1, epochs + 1):
        train = run_epoch(model, train_loader, device, optimizer)
        with torch.no_grad():
            val = run_epoch(model, val_loader, device)
        for partition, metrics in (("train", train), ("val", val)):
            history[f"{partition}_loss"].append(metrics["loss"])
            history[f"{partition}_accuracy"].append(metrics["accuracy"])
        
    return model, history

histories = {}
models = [
    ("EdgeMPNN", EdgeMPNNClassifier(
        hidden_dim=hidden,
        n_classes=len(classes)
    )),
    ("Global graph Transformer", GlobalGraphTransformerModel(
        node_dim=NODE_DIM,
        edge_dim=EDGE_DIM,
        hidden_dim=hidden,
        num_heads=4,
        num_transformer_layers=2,
        n_classes=len(classes)
    ))
]
results = []
for name, model in models:
        print(f"\nTraining {name}; parameters: {sum(p.numel() for p in model.parameters()):,}")
        model, history = train_model(model, train_loader, val_loader, device,
                                     epochs=epochs)
        with torch.no_grad():
            val = run_epoch(model, val_loader, device)
        print(f"{name}: val accuracy={val['accuracy']:.3f}, macro-F1={val['f1_macro']:.3f}")
        print(classification_report(val["y_true"], val["y_pred"],
                                    labels=list(range(len(classes))), target_names=classes,
                                    zero_division=0))
        results.append({"model": name, "val_accuracy": val["accuracy"],
                        "val_macro_f1": val["f1_macro"], "val_cross_entropy": val["loss"]})
        histories[name] =  history


Training GCN; parameters: 2,981


# Plot training progress

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for name, hist in histories.items():
    ep = np.arange(1, len(hist["train_loss"]) + 1)
    axes[0].plot(ep, hist["train_loss"], label=f"{name} train")
    axes[0].plot(ep, hist["val_loss"], "--", label=f"{name} validation")
    axes[1].plot(ep, hist["train_accuracy"], label=f"{name} train")
    axes[1].plot(ep, hist["val_accuracy"], "--", label=f"{name} validation")
for ax, title, ylabel in zip(axes, ["Cross-entropy loss", "Classification accuracy"],
                        ["Cross-entropy", "Accuracy"]):
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
fig.tight_layout()
    